# TSA — Week 2 Python Lab
## Explore time-series patterns using Phnom Penh precipitation

**Student:** __________________  |  **Group:** __________  |  **GitHub repository:** __________________

**Learning outcomes:** check the time index and monthly coverage; make a time plot; compare years in a seasonal plot; make a *real seasonal subseries plot*; support interpretations with evidence.

**Time:** ~75–90 minutes. Run cells in order, discuss the questions with a partner, and write answers in the blank Markdown answer cells. No ARIMA or forecasting model is required this week.

**Evidence and source caution.** The numeric table below is an *exact copy* of the CSV you uploaded (2015–2025). That CSV contains only `YEAR`, 12 month columns and `ANN`: its NASA parameter name, spatial metadata and units header are **not included**. The filename encodes an approximate location near 11.56° N, 104.93° E. Based on the proposed NASA POWER request, its presumed parameter is `PRECTOTCORR_SUM` (corrected precipitation sum). **Confirm parameter, location, and precipitation units using the full NASA download header/metadata before using this in a publication.** No values are invented or replaced. These are *gridded estimates*, not a local gauge measurement.

Official documentation: [NASA POWER monthly API](https://power.larc.nasa.gov/docs/services/api/temporal/monthly/) · [NASA POWER parameter dictionary](https://power.larc.nasa.gov/docs/tutorials/parameters/).

**Textbook basis:** Hyndman & Athanasopoulos, *Forecasting: Principles and Practice*, Ch. 2 §§2.1–2.5 (time plots, time-series patterns, seasonal plots and seasonal subseries plots); Shumway & Stoffer, *Time Series Analysis and Its Applications*, 5th ed., §1.1. Python/Colab and the Cambodia data are classroom adaptations.

### 0 · Dataset (included so this notebook runs immediately in Colab)
The exact uploaded CSV is embedded below. You do **not** need to download another file or connect to Google Drive. We retain the source `ANN` column only to cross-check annual totals; we will NOT treat it as a 13th month.

In [ ]:
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Exact table copied from the uploaded NASA POWER-named CSV; no interpolation.
RAW_CSV = 'YEAR,JAN,FEB,MAR,APR,MAY,JUN,JUL,AUG,SEP,OCT,NOV,DEC,ANN\n2015,3.26,6.85,6.79,86.97,46.84,85.88,41.22,288.64,133.08,134.41,70.89,13.06,917.89\n2016,26.66,4.83,6.41,7.81,117.14,43.76,25.41,23.91,138.2,166.56,137.19,33.45,731.33\n2017,18.64,16.04,14.73,29.46,123.78,62.85,140.62,175,213.37,87.4,93.56,41.91,1017.36\n2018,75.61,7.22,80.34,94.97,177.46,338.51,205.87,41.17,224.33,155.88,20.71,39.95,1462.02\n2019,19.07,2.88,25.55,59.37,236.16,392.3,223.72,263.68,263.18,93.11,29.63,0.34,1608.99\n2020,26.58,2.14,34.14,146.86,92.3,703.62,166.68,174.17,280.88,496.85,98.6,67.73,2290.55\n2021,11.54,10.8,10.14,111.31,243.46,121.11,299.33,261.06,329.28,240.56,213,31.5,1883.09\n2022,5.36,56.69,128.22,115.23,205.65,262.05,439.12,276.57,250.76,185.5,130.42,26.53,2082.1\n2023,30.18,6.23,1.08,22.04,108.3,154.58,177.06,158.74,315.42,266.52,131.32,30.82,1402.29\n2024,1.75,0.8,13.03,2.15,146.8,212.52,271.85,147.87,207.61,233.95,50.78,20.45,1309.56\n2025,1.02,19.62,44.45,48.73,46.49,69.32,36.31,291.32,146,172.7,91.83,16.72,984.51'
raw = pd.read_csv(io.StringIO(RAW_CSV))
print('Source table:', raw.shape[0], 'years ×', len(raw.columns)-2, 'months')
display(raw.head(3))

### 1 · Data detective (10 minutes)
**Partner discussion:** What is our time index? Target? Frequency? What does `ANN` represent? Is the row for 2020 one observation or twelve observations?

Run the next cell. Do not silently fill missing dates or replace unusual numbers.

In [ ]:
MONTHS = ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']
assert list(raw.columns) == ['YEAR', *MONTHS, 'ANN'], 'Unexpected file structure'
assert raw['YEAR'].is_unique, 'Duplicate years in the source'

# Wide (one year per row) -> long (one calendar month per row).
long = raw.melt(id_vars=['YEAR'], value_vars=MONTHS,
                var_name='month_name', value_name='precipitation')
month_to_number = {m: i for i, m in enumerate(MONTHS, start=1)}
long['month'] = long['month_name'].map(month_to_number)
long['date'] = pd.to_datetime(dict(year=long.YEAR, month=long.month, day=1))
long = long[['date','YEAR','month','month_name','precipitation']].sort_values('date').reset_index(drop=True)
long['precipitation'] = pd.to_numeric(long['precipitation'], errors='coerce')
long.loc[long['precipitation'] <= -900, 'precipitation'] = np.nan  # NASA missing-value flag if present

expected = pd.date_range('2015-01-01','2025-12-01',freq='MS')
missing_dates = expected.difference(pd.DatetimeIndex(long['date']))
duplicate_dates = long['date'].duplicated().sum()
missing_values = long['precipitation'].isna().sum()
annual_check = (raw[MONTHS].sum(axis=1) - raw['ANN']).abs()

print('Range:', long.date.min().date(), 'to', long.date.max().date())
print('Observations:', len(long), '| Expected:', len(expected))
print('Missing months:', len(missing_dates), '| Duplicate dates:', duplicate_dates,
      '| Missing values:', missing_values)
print('Largest absolute difference between monthly sum and ANN:', round(annual_check.max(), 3))
display(long.head(4))

assert len(long) == len(expected) and len(missing_dates) == 0, 'Monthly coverage is incomplete'
assert duplicate_dates == 0 and missing_values == 0, 'Investigate duplicates/missing values'
assert annual_check.max() < 0.1, 'Investigate annual total inconsistency' 

**Answer 1 (write 2–3 sentences):** Identify the time index, target, frequency, date range and missing-data result. State that the measurement unit is *pending independent confirmation from full source metadata*.

> Your answer: …

### 2 · Time plot (10 minutes)
A time plot shows observations in chronological order. Look for long-term movement, repeating patterns and unusual observations (FPP §2.2).

In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(long['date'], long['precipitation'], marker='.', markersize=4, linewidth=1.4)
ax.set(title='Phnom Penh monthly precipitation (2015–2025)', xlabel='Time',
       ylabel='Precipitation (source units; confirm mm)')
ax.grid(alpha=.25)
plt.tight_layout()
plt.show()

**Answer 2:** Describe (a) any repeating rise/fall, (b) whether a long-term increase/decrease is clear, and (c) a specific unusual observation. Mention its **month and year**, not a guessed cause.

> Your answer: …

### 3 · Seasonal plot (15 minutes)
A seasonal plot places month on the horizontal axis and draws a separate line for each year, so we can compare the same calendar month across years (FPP §2.4). The data below are *exactly the same observations* as in the time plot.

In [ ]:
fig, ax = plt.subplots(figsize=(12,5))
for year, part in long.groupby('YEAR', sort=True):
    ax.plot(part['month'], part['precipitation'], marker='.', linewidth=1.2, label=str(year))
ax.set_xticks(range(1,13), MONTHS)
ax.set(title='Seasonal plot — same monthly data, one line per year', xlabel='Month',
       ylabel='Precipitation (source units; confirm mm)')
ax.grid(alpha=.2)
ax.legend(title='Year', bbox_to_anchor=(1.01,1), loc='upper left', ncol=1, fontsize=8)
plt.tight_layout()
plt.show()

**Answer 3:** Which months tend to be wetter, according to the graph? Is the peak the same size each year? Which year has an unusual monthly spike? Avoid saying that a particular event *caused* the spike without supporting evidence.

> Your answer: …

### 4 · Seasonal subseries plot (15 minutes)
**Important distinction:** a seasonal subseries plot is **not** just a scatterplot of dots grouped by month. FPP §2.5 arranges each month’s observations across years in its own mini time plot and often draws a horizontal line at that month’s mean. We use the same global vertical axis to help compare months.

In [ ]:
month_mean = long.groupby('month')['precipitation'].mean()
ymax = long['precipitation'].max()*1.08
fig, axes = plt.subplots(2, 6, figsize=(16,6), sharex=True, sharey=True)
for m, ax in enumerate(axes.flat, start=1):
    part = long[long['month'] == m].sort_values('YEAR')
    ax.plot(part['YEAR'], part['precipitation'], marker='o', markersize=3, linewidth=1)
    ax.axhline(month_mean.loc[m], linestyle='--', linewidth=1, label='Month mean')
    ax.set(title=MONTHS[m-1], xlim=(2014.6, 2025.4), ylim=(0,ymax))
    ax.set_xticks([2015,2020,2025])
    ax.tick_params(axis='x', labelrotation=45)
    ax.grid(alpha=.15)
for ax in axes[:,0]: ax.set_ylabel('Precipitation (source units)')
fig.suptitle('Seasonal subseries — each month across years; dashed line = that month’s mean', y=1.01)
plt.tight_layout()
plt.show()
print('Mean precipitation by calendar month (source units):')
display(pd.DataFrame({'Month':MONTHS, 'Mean':month_mean.reindex(range(1,13)).round(2).values}))

**Answer 4:** Identify one month with a high average and one with a low average. Compare how much observations vary across years for those two months. Name the **specific month** containing the largest value; does that make it an error? Why or why not?

> Your answer: …

### 5 · Evidence challenge (10 minutes)
Choose one claim. Use a number from the table or a feature visible in a plot as evidence. Seasonal patterns do **not** mean values must be identical every year. An unusual value is not automatically a measurement error.

In [ ]:
max_row = long.loc[long['precipitation'].idxmax(), ['date','precipitation']]
print('Largest observed monthly value:',max_row['date'].strftime('%Y-%m'), '=', max_row['precipitation'])
print('Month with highest mean:', MONTHS[int(month_mean.idxmax())-1])
print('Month with lowest mean:', MONTHS[int(month_mean.idxmin())-1])
print('NOTE: these are observations and descriptive summaries, NOT forecasts.')

**Answer 5:** Write a 3–4 sentence evidence-based summary of the pattern. Mention at least one limitation (e.g. source metadata needs checking, unusual year, or patterns may change).

> Your answer: …

### 6 · Optional mini-challenge: Which view answers which question? (5 minutes)
Match each question to a plot. Discuss before writing your answer.

- Where does the series move over the **whole 11 years**?
- How do **July values** compare between years?
- Which months have high **average** precipitation, and how does one month vary over time?

> Your match and justification: …

### 7 · Save evidence to your semester project (10 minutes)
Run the cell below to export the **reshaped copy** and three figures. Download and add these files, the notebook, and a short write-up to your GitHub repository. Keep the original uploaded file unchanged. Do not upload private data.

Suggested structure: `data/` (with source note), `notebooks/`, `figures/`, `LEARNING_LOG.md`.

In [ ]:
from pathlib import Path
output = Path('week2_outputs')
output.mkdir(exist_ok=True)
long.to_csv(output/'phnom_penh_monthly_precipitation_2015_2025_long.csv', index=False)

fig1, ax1 = plt.subplots(figsize=(10,3.5))
ax1.plot(long.date, long.precipitation, linewidth=1.2)
ax1.set(xlabel='Time', ylabel='Precipitation (source units)', title='Time plot')
fig1.tight_layout(); fig1.savefig(output/'01_time_plot.png', dpi=160); plt.close(fig1)

fig2, ax2 = plt.subplots(figsize=(10,4))
for year, part in long.groupby('YEAR'):
    ax2.plot(part.month,part.precipitation,linewidth=1,label=str(year))
ax2.set_xticks(range(1,13), MONTHS)
ax2.set(xlabel='Month',ylabel='Precipitation (source units)',title='Seasonal plot')
ax2.legend(title='Year',bbox_to_anchor=(1.01,1),loc='upper left',fontsize=7)
fig2.tight_layout();fig2.savefig(output/'02_seasonal_plot.png',dpi=160,bbox_inches='tight');plt.close(fig2)

fig3,axs=plt.subplots(2,6,figsize=(16,6),sharex=True,sharey=True)
for m,ax in enumerate(axs.flat,1):
    part=long[long.month==m]
    ax.plot(part.YEAR,part.precipitation,marker='.',linewidth=1)
    ax.axhline(month_mean.loc[m],linestyle='--',linewidth=1)
    ax.set(title=MONTHS[m-1],xlim=(2014.6,2025.4),ylim=(0,ymax))
    ax.set_xticks([2015,2020,2025]); ax.tick_params(axis='x',labelrotation=45)
fig3.suptitle('Seasonal subseries plot')
fig3.tight_layout();fig3.savefig(output/'03_seasonal_subseries_plot.png',dpi=160,bbox_inches='tight');plt.close(fig3)

print('Created:', ', '.join(str(p) for p in sorted(output.iterdir())))
try:
    from google.colab import files
    import shutil
    zip_path = shutil.make_archive('tsa_week2_outputs','zip',root_dir=str(output))
    print('Output ZIP:',zip_path)
    # Uncomment the next line to download the ZIP in Google Colab.
    # files.download(zip_path)
except ImportError:
    print('In Colab, uncomment files.download(zip_path) to download all figures and data.')

### Submission checklist
- [ ] All notebook cells run without errors.
- [ ] Answer cells 1–5 contain **your own** evidence-based explanations.
- [ ] Include all three plots in your GitHub `figures/` folder.
- [ ] Keep the original CSV or source citation and explain that the downloaded table lacks NASA metadata.
- [ ] Put the notebook under `notebooks/` and update `LEARNING_LOG.md`.
- [ ] Share **one GitHub link** (and one annotated graph in your Miro group frame).

**Exit ticket:** What is the difference between a seasonal plot and seasonal subseries plot? What remains unclear?

**Next week:** lag relationships / autocorrelation — today’s plots are exploration, not evidence that a forecast will be accurate.